In [10]:
# This notebook answers RQ2: Does the percentage of hiking routes per difficulty level relate to the mean elevation of a canton?

import pandas as pd
# pandas: works with tables (DataFrames).
# Used to load the hiking routes and the canton figures and to calculate percentages per canton.

%pip install scipy
#is required for calculating correlations

   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------- ----------------------- 14.9/36.6 MB 88.9 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 104.2 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 88.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load the hiking routes with cantons (result of canton_assignment.ipynb)
hiking_routes = pd.read_csv("old_csv/result_wanderland_cantons.csv")

# Check: number of routes
print("Hiking routes:", len(hiking_routes))

Hiking routes: 274


In [3]:
# Check: which difficulty levels occur, and how often
print(hiking_routes["technical"].value_counts(dropna=False))

technical
mittel (Bergwanderweg)    161
leicht (Wanderweg)        108
mittel (Wanderweg)          3
leicht                      2
Name: count, dtype: int64


In [4]:
# Check: which routes have the difficulty "leicht" or "mittel (Wanderweg)"?
hiking_routes.loc[
    hiking_routes["technical"].isin(["leicht", "mittel (Wanderweg)"]),
    ["title", "technical", "url"]
]

,title,technical,url
6,Tour de La Palette,mittel (Wanderweg),https://schweizmobil.ch/de/wanderland/route-109
61,Sentier des alpages de Vionnaz,leicht,https://schweizmobil.ch/de/wanderland/route-225
64,Le Grand Tour des Vanils,mittel (Wanderweg),https://schweizmobil.ch/de/wanderland/route-263
114,Chemin Tiergart-Plain Fayen,leicht,https://schweizmobil.ch/de/wanderland/route-451
116,Chemin du Val Terbi,mittel (Wanderweg),https://schweizmobil.ch/de/wanderland/route-454


In [5]:
# One row per route and canton (a route counts for every canton in which it starts or ends),
# together with the difficulty level of the route.
route_canton = pd.concat([
    hiking_routes[["start_canton", "technical"]].rename(columns={"start_canton": "canton"}),
    hiking_routes[["end_canton", "technical"]].rename(columns={"end_canton": "canton"}),
])

# A route that starts and ends in the same canton counts only once for that canton
route_canton = route_canton.reset_index().drop_duplicates()

# Routes abroad are not counted
route_canton = route_canton[route_canton["canton"] != "abroad"]

# Check: number of route-canton combinations
print("Route-canton combinations:", len(route_canton))

Route-canton combinations: 279


In [6]:
# Percentage of each difficulty level per canton
# crosstab counts the routes for each combination of canton and difficulty level;
# normalize="index" turns the counts into shares per canton (each row adds up to 100 %).
difficulty_share = (pd.crosstab(route_canton["canton"], route_canton["technical"], normalize="index") * 100).round(1)

# Add the total number of routes per canton, to see on how many routes each percentage is based
difficulty_share["routes"] = route_canton["canton"].value_counts()

print(difficulty_share.to_string())

technical  leicht  leicht (Wanderweg)  mittel (Bergwanderweg)  mittel (Wanderweg)  routes
canton                                                                                   
AI            0.0                50.0                    50.0                 0.0       2
AR            0.0                60.0                    40.0                 0.0       5
BE            0.0                34.3                    65.7                 0.0      35
BL            0.0                75.0                    25.0                 0.0       4
FR            0.0                60.0                    33.3                 6.7      15
GE            0.0                83.3                    16.7                 0.0       6
GL            0.0                45.5                    54.5                 0.0      11
GR            0.0                12.0                    88.0                 0.0      50
JU           14.3                57.1                    14.3                14.3       7
LU        

In [7]:
# Load the geographic key figures of the cantons (BFS, as of 01.01.2026)
canton_figures = pd.read_excel("Geographische_Kennzahlen.xlsx", sheet_name="Daten")

# Canton abbreviation for each official canton number (order of the Federal Constitution)
canton_abbreviations = {
    1: "ZH", 2: "BE", 3: "LU", 4: "UR", 5: "SZ", 6: "OW", 7: "NW", 8: "GL", 9: "ZG",
    10: "FR", 11: "SO", 12: "BS", 13: "BL", 14: "SH", 15: "AR", 16: "AI", 17: "SG",
    18: "GR", 19: "AG", 20: "TG", 21: "TI", 22: "VD", 23: "VS", 24: "NE", 25: "GE", 26: "JU"
}
canton_figures["canton"] = canton_figures["KTNR"].map(canton_abbreviations)

# Check: abbreviation, name and mean elevation (Z_AVG, in metres above sea level)
canton_figures[["canton", "KTNAME", "Z_AVG"]].head()

,canton,KTNAME,Z_AVG
0,ZH,Zürich,533
1,BE,Bern / Berne,1200
2,LU,Luzern,770
3,UR,Uri,1899
4,SZ,Schwyz,1096


In [8]:
# Combine the difficulty shares with the mean elevation of each canton.
# Only cantons with hiking routes are included (AG, BS and ZG have none).
rq2 = difficulty_share.reset_index().merge(
    canton_figures[["canton", "Z_AVG"]],
    on="canton",
    how="left"
)

# Sort by mean elevation, from the highest to the lowest canton
rq2 = rq2.sort_values("Z_AVG", ascending=False).reset_index(drop=True)

# Check: canton, mean elevation, share of mountain hiking trails and number of routes
print(rq2[["canton", "Z_AVG", "mittel (Bergwanderweg)", "routes"]].to_string())

   canton  Z_AVG  mittel (Bergwanderweg)  routes
0      VS   2135                    84.4      32
1      GR   2020                    88.0      50
2      UR   1899                   100.0       6
3      GL   1579                    54.5      11
4      TI   1391                    87.5      16
5      OW   1337                    83.3       6
6      BE   1200                    65.7      35
7      AI   1128                    50.0       2
8      SZ   1096                    55.6       9
9      NW   1071                    80.0      10
10     SG    982                    64.3      14
11     AR    929                    40.0       5
12     NE    912                    25.0       4
13     FR    855                    33.3      15
14     VD    841                    12.0      25
15     LU    770                     0.0       2
16     JU    690                    14.3       7
17     SO    616                    16.7       6
18     SH    537                     0.0       5
19     ZH    533    

In [12]:
from scipy.stats import spearmanr
# scipy: library for scientific calculations.
# spearmanr calculates the Spearman correlation together with its p-value.

result = spearmanr(rq2["Z_AVG"], rq2["mittel (Bergwanderweg)"])
print("Spearman correlation:", round(result.statistic, 2))
print("p-value:", result.pvalue)

Spearman correlation: 0.88
p-value: 2.2256235784032833e-08


In [13]:
# Save the result of RQ2: difficulty shares and mean elevation per canton
rq2.to_csv("rq2_result_difficulty_elevation.csv", index=False)
print("Saved:", len(rq2), "cantons")

Saved: 23 cantons


In [1]:
import pandas as pd
import rq2
from utils import load_canton_figures

# Test only: the four route tables with cantons replace the pipeline for now
test_routes = pd.concat([
    pd.read_csv("old_csv/result_wanderland_cantons.csv"),
    pd.read_csv("old_csv/result_veloland_cantons.csv"),
    pd.read_csv("old_csv/result_mountainbikeland_cantons.csv"),
    pd.read_csv("old_csv/result_skatingland_cantons.csv"),
], ignore_index=True)

table, correlation = rq2.run_rq2(test_routes, load_canton_figures())
print("Cantons:", len(table))
print("Spearman:", round(correlation.statistic, 2), "p-value:", correlation.pvalue)

Spearman correlation: 0.88
p-value: 2.225623578403286e-08
Cantons: 23
Spearman: 0.88 p-value: 2.225623578403286e-08
